# 감정인식 프로젝트 - 데이터 준비

EDA에서 실제 학습에 사용할 데이터로 TRAIN_01과 TRAIN_02를 선택하였다.

이 노트북에서는 선택한 원천 이미지와 라벨 데이터를 실제 학습에 사용할 수 있도록 준비한다.

주요 작업:
- 프로젝트 경로 설정
- TRAIN_01 / TRAIN_02 원천 이미지 압축 해제
- 라벨 데이터 준비
- 이미지와 라벨 구조 정리
- 압축 해제 및 데이터 구성 결과 확인

resize, 흑백 변환, 얼굴 crop, 모델별 preprocessing 등은
모델 및 학습 방향을 결정한 후 진행한다.

## 1. 라이브러리 및 프로젝트 경로 설정

In [4]:
# 파일 경로와 ZIP 파일을 다루기 위한 라이브러리
from pathlib import Path
import zipfile

# 프로젝트의 가장 상위 폴더
PROJECT_ROOT = Path(r"D:\emotion_recognition_project")

# 데이터 폴더와 다운로드한 ZIP 파일 폴더
DATA_DIR = PROJECT_ROOT / "02_data"
DOWNLOAD_DIR = DATA_DIR / "downloads"

# 경로가 올바른지 확인
print("프로젝트 경로:", PROJECT_ROOT)
print("다운로드 경로:", DOWNLOAD_DIR)

프로젝트 경로: D:\emotion_recognition_project
다운로드 경로: D:\emotion_recognition_project\02_data\downloads


In [5]:
# 현재 02_data 폴더 구조 확인
for path in DATA_DIR.iterdir():
    print(path.name)

downloads
labels
raw
sample


In [8]:
# 압축 해제할 폴더 경로
RAW_DIR = DATA_DIR / "raw"
LABEL_DIR = DATA_DIR / "labels"

TRAIN_RAW_DIR = RAW_DIR / "train"
VALID_RAW_DIR = RAW_DIR / "valid"

TRAIN_LABEL_DIR = LABEL_DIR / "train"
VALID_LABEL_DIR = LABEL_DIR / "valid"

# 폴더가 없으면 생성
TRAIN_RAW_DIR.mkdir(parents=True, exist_ok=True)
VALID_RAW_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_LABEL_DIR.mkdir(parents=True, exist_ok=True)
VALID_LABEL_DIR.mkdir(parents=True, exist_ok=True)

print("Training 원천 이미지 폴더:", TRAIN_RAW_DIR)
print("Training 라벨 폴더:", TRAIN_LABEL_DIR)

Training 원천 이미지 폴더: D:\emotion_recognition_project\02_data\raw\train
Training 라벨 폴더: D:\emotion_recognition_project\02_data\labels\train


In [19]:
import shutil

# 테스트로 만든 기쁨 TRAIN_01 폴더
JOY_TRAIN01_DIR = TRAIN_RAW_DIR / "EMOIMG_기쁨_TRAIN_01"

# 폴더 안의 이미지들을 raw/train으로 이동
for file_path in JOY_TRAIN01_DIR.iterdir():

    if file_path.is_file():

        target_path = TRAIN_RAW_DIR / file_path.name

        # 같은 이름의 파일이 이미 있는지 확인
        if target_path.exists():
            print("중복 파일 발견:", file_path.name)
        else:
            shutil.move(
                str(file_path),
                str(target_path)
            )

print("기쁨 TRAIN_01 이미지 이동 완료")

기쁨 TRAIN_01 이미지 이동 완료


## 2. 학습 및 검증 데이터 압축 해제

EDA에서 실제 학습에 사용할 Training 데이터로
7개 감정 클래스의 TRAIN_01과 TRAIN_02를 선택하였다.

선택한 Training 원천데이터와 전체 Validation 원천데이터를
실제 학습에 사용할 수 있도록 압축 해제한다.

### 2-1. Training 압축 해제 대상 확인

In [6]:
# downloads 폴더 아래의 모든 ZIP 파일 찾기
all_zip_files = list(DOWNLOAD_DIR.rglob("*.zip"))

# Training 원천데이터 중 TRAIN_01, TRAIN_02만 선택
train_source_zips = [
    path for path in all_zip_files
    if (
        "[원천]" in path.name
        and ("TRAIN_01" in path.name or "TRAIN_02" in path.name)
    )
]

# Training 라벨 ZIP만 선택
train_label_zips = [
    path for path in all_zip_files
    if (
        "[라벨]" in path.name
        and "TRAIN" in path.name
    )
]

print("선택된 Training 원천 ZIP 수:", len(train_source_zips))
print("선택된 Training 라벨 ZIP 수:", len(train_label_zips))

선택된 Training 원천 ZIP 수: 14
선택된 Training 라벨 ZIP 수: 7


In [7]:
# 선택된 Training 원천 ZIP 목록 확인
print("=== Training 원천 ZIP ===")

for path in sorted(train_source_zips):
    print(path.name)

print("\n=== Training 라벨 ZIP ===")

for path in sorted(train_label_zips):
    print(path.name)

=== Training 원천 ZIP ===
[원천]EMOIMG_기쁨_TRAIN_01.zip
[원천]EMOIMG_기쁨_TRAIN_02.zip
[원천]EMOIMG_당황_TRAIN_01.zip
[원천]EMOIMG_당황_TRAIN_02.zip
[원천]EMOIMG_분노_TRAIN_01.zip
[원천]EMOIMG_분노_TRAIN_02.zip
[원천]EMOIMG_불안_TRAIN_01.zip
[원천]EMOIMG_불안_TRAIN_02.zip
[원천]EMOIMG_상처_TRAIN_01.zip
[원천]EMOIMG_상처_TRAIN_02.zip
[원천]EMOIMG_슬픔_TRAIN_01.zip
[원천]EMOIMG_슬픔_TRAIN_02.zip
[원천]EMOIMG_중립_TRAIN_01.zip
[원천]EMOIMG_중립_TRAIN_02.zip

=== Training 라벨 ZIP ===
[라벨]EMOIMG_기쁨_TRAIN.zip
[라벨]EMOIMG_당황_TRAIN.zip
[라벨]EMOIMG_분노_TRAIN.zip
[라벨]EMOIMG_불안_TRAIN.zip
[라벨]EMOIMG_상처_TRAIN.zip
[라벨]EMOIMG_슬픔_TRAIN.zip
[라벨]EMOIMG_중립_TRAIN.zip


### 2-2. ZIP 한글 파일명 복원 확인

In [9]:
# 테스트할 원천 ZIP 1개 선택
test_zip = sorted(train_source_zips)[0]

print("테스트 ZIP:", test_zip.name)

테스트 ZIP: [원천]EMOIMG_기쁨_TRAIN_01.zip


In [10]:
# ZIP 내부에서 깨진 한글 파일명을 복원하는 함수
def fix_zip_filename(name):
    try:
        return name.encode("cp437").decode("cp949")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return name

### 2-3. Training 원천 이미지 압축 해제 및 검증

In [11]:
# 테스트 ZIP 내부 파일명 확인
with zipfile.ZipFile(test_zip, "r") as z:
    for member in z.namelist()[:3]:

        # ZIP 내부에서 깨진 한글 파일명 복원
        fixed_member = fix_zip_filename(member)

        print("원본 :", member)
        print("복원 :", fixed_member)
        print()

원본 : EMOIMG_▒Γ╗▌_TRAIN_01/
복원 : EMOIMG_기쁨_TRAIN_01/

원본 : EMOIMG_▒Γ╗▌_TRAIN_01/006b56dc2f8cda2361e1b01b2496d6f352dd5b1790f0a9b0bfcbe540b292247d_┐⌐_20_▒Γ╗▌_░°░°╜├╝│&┴╛▒│&└╟╖ß╜├╝│_20210130213913-001-001.jpg
복원 : EMOIMG_기쁨_TRAIN_01/006b56dc2f8cda2361e1b01b2496d6f352dd5b1790f0a9b0bfcbe540b292247d_여_20_기쁨_공공시설&종교&의료시설_20210130213913-001-001.jpg

원본 : EMOIMG_▒Γ╗▌_TRAIN_01/006b56dc2f8cda2361e1b01b2496d6f352dd5b1790f0a9b0bfcbe540b292247d_┐⌐_20_▒Γ╗▌_░°░°╜├╝│&┴╛▒│&└╟╖ß╜├╝│_20210130213913-001-002.jpg
복원 : EMOIMG_기쁨_TRAIN_01/006b56dc2f8cda2361e1b01b2496d6f352dd5b1790f0a9b0bfcbe540b292247d_여_20_기쁨_공공시설&종교&의료시설_20210130213913-001-002.jpg



In [12]:
# 복원된 한글 파일명으로 테스트 ZIP 1개 압축 해제
with zipfile.ZipFile(test_zip, "r") as z:

    for member in z.namelist():

        # 깨진 한글 경로/파일명 복원
        fixed_member = fix_zip_filename(member)

        # 저장할 최종 경로
        target_path = TRAIN_RAW_DIR / fixed_member

        # 폴더인 경우 생성
        if member.endswith("/"):
            target_path.mkdir(
                parents=True,
                exist_ok=True
            )
            continue

        # 파일이 들어갈 상위 폴더 생성
        target_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        # ZIP 내부 파일 읽기
        with z.open(member) as source:
            with open(target_path, "wb") as target:
                target.write(source.read())

print("테스트 압축 해제 완료:", test_zip.name)

테스트 압축 해제 완료: [원천]EMOIMG_기쁨_TRAIN_01.zip


In [13]:
# 압축 해제된 폴더 확인
for path in TRAIN_RAW_DIR.iterdir():
    print(path.name)

EMOIMG_기쁨_TRAIN_01


In [14]:
# 압축 해제된 이미지 파일명 일부 확인
test_extract_dir = TRAIN_RAW_DIR / "EMOIMG_기쁨_TRAIN_01"

for path in list(test_extract_dir.iterdir())[:3]:
    print(path.name)

006b56dc2f8cda2361e1b01b2496d6f352dd5b1790f0a9b0bfcbe540b292247d_여_20_기쁨_공공시설&종교&의료시설_20210130213913-001-001.jpg
006b56dc2f8cda2361e1b01b2496d6f352dd5b1790f0a9b0bfcbe540b292247d_여_20_기쁨_공공시설&종교&의료시설_20210130213913-001-002.jpg
006b56dc2f8cda2361e1b01b2496d6f352dd5b1790f0a9b0bfcbe540b292247d_여_20_기쁨_공공시설&종교&의료시설_20210130213913-001-004.jpg


In [15]:
# 테스트 ZIP을 제외한 나머지 Training 원천 ZIP 압축 해제
remaining_source_zips = [
    zip_path for zip_path in train_source_zips
    if zip_path != test_zip
]

print("남은 원천 ZIP 수:", len(remaining_source_zips))

남은 원천 ZIP 수: 13


In [16]:
# 나머지 Training 원천 ZIP 13개 압축 해제
for zip_path in sorted(remaining_source_zips):

    print("압축 해제 중:", zip_path.name)

    with zipfile.ZipFile(zip_path, "r") as z:

        for member in z.namelist():

            # 깨진 한글 경로와 파일명 복원
            fixed_member = fix_zip_filename(member)

            # 저장할 경로 설정
            target_path = TRAIN_RAW_DIR / fixed_member

            # 폴더인 경우 생성
            if member.endswith("/"):
                target_path.mkdir(
                    parents=True,
                    exist_ok=True
                )
                continue

            # 파일이 들어갈 상위 폴더 생성
            target_path.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            # ZIP 내부 파일을 실제 파일로 저장
            with z.open(member) as source:
                with open(target_path, "wb") as target:
                    target.write(source.read())

    print("완료:", zip_path.name)

print("Training 원천 이미지 압축 해제 완료")

압축 해제 중: [원천]EMOIMG_기쁨_TRAIN_02.zip
완료: [원천]EMOIMG_기쁨_TRAIN_02.zip
압축 해제 중: [원천]EMOIMG_당황_TRAIN_01.zip
완료: [원천]EMOIMG_당황_TRAIN_01.zip
압축 해제 중: [원천]EMOIMG_당황_TRAIN_02.zip
완료: [원천]EMOIMG_당황_TRAIN_02.zip
압축 해제 중: [원천]EMOIMG_분노_TRAIN_01.zip
완료: [원천]EMOIMG_분노_TRAIN_01.zip
압축 해제 중: [원천]EMOIMG_분노_TRAIN_02.zip
완료: [원천]EMOIMG_분노_TRAIN_02.zip
압축 해제 중: [원천]EMOIMG_불안_TRAIN_01.zip
완료: [원천]EMOIMG_불안_TRAIN_01.zip
압축 해제 중: [원천]EMOIMG_불안_TRAIN_02.zip
완료: [원천]EMOIMG_불안_TRAIN_02.zip
압축 해제 중: [원천]EMOIMG_상처_TRAIN_01.zip
완료: [원천]EMOIMG_상처_TRAIN_01.zip
압축 해제 중: [원천]EMOIMG_상처_TRAIN_02.zip
완료: [원천]EMOIMG_상처_TRAIN_02.zip
압축 해제 중: [원천]EMOIMG_슬픔_TRAIN_01.zip
완료: [원천]EMOIMG_슬픔_TRAIN_01.zip
압축 해제 중: [원천]EMOIMG_슬픔_TRAIN_02.zip
완료: [원천]EMOIMG_슬픔_TRAIN_02.zip
압축 해제 중: [원천]EMOIMG_중립_TRAIN_01.zip
완료: [원천]EMOIMG_중립_TRAIN_01.zip
압축 해제 중: [원천]EMOIMG_중립_TRAIN_02.zip
완료: [원천]EMOIMG_중립_TRAIN_02.zip
Training 원천 이미지 압축 해제 완료


In [17]:
# 압축 해제된 Training 폴더 수 확인
train_folders = [
    path for path in TRAIN_RAW_DIR.iterdir()
    if path.is_dir()
]

print("Training 폴더 수:", len(train_folders))

for path in sorted(train_folders):
    print(path.name)

Training 폴더 수: 1
EMOIMG_기쁨_TRAIN_01


In [20]:
# 기쁨 TRAIN_01 폴더에 남은 파일 수 확인
print(
    "남아 있는 파일 수:",
    len(list(JOY_TRAIN01_DIR.iterdir()))
)

남아 있는 파일 수: 0


In [21]:
# raw/train 바로 아래의 이미지 파일 수 확인
image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}

train_image_count = sum(
    1
    for path in TRAIN_RAW_DIR.iterdir()
    if path.is_file()
    and path.suffix.lower() in image_extensions
)

print("Training 이미지 수:", train_image_count)

Training 이미지 수: 223578


### 2-4. Training 라벨 압축 해제 및 검증

In [22]:
# Training 라벨 ZIP 7개 압축 해제
for zip_path in sorted(train_label_zips):

    print("압축 해제 중:", zip_path.name)

    with zipfile.ZipFile(zip_path, "r") as z:

        for member in z.namelist():

            # 깨진 한글 경로와 파일명 복원
            fixed_member = fix_zip_filename(member)

            # 저장할 경로 설정
            target_path = TRAIN_LABEL_DIR / fixed_member

            # 폴더인 경우 생성
            if member.endswith("/"):
                target_path.mkdir(
                    parents=True,
                    exist_ok=True
                )
                continue

            # 파일이 들어갈 상위 폴더 생성
            target_path.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            # ZIP 내부 파일을 실제 파일로 저장
            with z.open(member) as source:
                with open(target_path, "wb") as target:
                    target.write(source.read())

    print("완료:", zip_path.name)

print("Training 라벨 압축 해제 완료")

압축 해제 중: [라벨]EMOIMG_기쁨_TRAIN.zip
완료: [라벨]EMOIMG_기쁨_TRAIN.zip
압축 해제 중: [라벨]EMOIMG_당황_TRAIN.zip
완료: [라벨]EMOIMG_당황_TRAIN.zip
압축 해제 중: [라벨]EMOIMG_분노_TRAIN.zip
완료: [라벨]EMOIMG_분노_TRAIN.zip
압축 해제 중: [라벨]EMOIMG_불안_TRAIN.zip
완료: [라벨]EMOIMG_불안_TRAIN.zip
압축 해제 중: [라벨]EMOIMG_상처_TRAIN.zip
완료: [라벨]EMOIMG_상처_TRAIN.zip
압축 해제 중: [라벨]EMOIMG_슬픔_TRAIN.zip
완료: [라벨]EMOIMG_슬픔_TRAIN.zip
압축 해제 중: [라벨]EMOIMG_중립_TRAIN.zip
완료: [라벨]EMOIMG_중립_TRAIN.zip
Training 라벨 압축 해제 완료


In [24]:
import json

# 각 Training 라벨 JSON 내부의 데이터 수 확인
total_label_records = 0

for json_path in sorted(TRAIN_LABEL_DIR.rglob("*.json")):

    with open(json_path, "r", encoding="utf-8") as f:
        label_data = json.load(f)

    record_count = len(label_data)
    total_label_records += record_count

    print(
        f"{json_path.name}: "
        f"{record_count:,}건"
    )

print(
    "\nTraining 전체 라벨 데이터 수:",
    f"{total_label_records:,}건"
)

img_emotion_training_data(기쁨).json: 60,103건
img_emotion_training_data(당황).json: 59,643건
img_emotion_training_data(분노).json: 59,696건
img_emotion_training_data(불안).json: 59,262건
img_emotion_training_data(상처).json: 59,389건
img_emotion_training_data(슬픔).json: 59,841건
img_emotion_training_data(중립).json: 59,233건

Training 전체 라벨 데이터 수: 417,167건


### 2-5. Validation 원천 이미지 압축 해제 및 검증

In [26]:
# Validation 원천 ZIP 7개 선택
valid_source_zips = [
    path for path in all_zip_files
    if (
        "VALID" in path.name
        and (
            "[원천]" in path.name
            or "원천데이터_0114_add" in str(path.parent)
        )
    )
]

print("선택된 Validation 원천 ZIP 수:", len(valid_source_zips))

for path in sorted(valid_source_zips):
    print(path.name)

선택된 Validation 원천 ZIP 수: 7
EMOIMG_기쁨_VALID.zip
[원천]EMOIMG_당황_VALID.zip
[원천]EMOIMG_분노_VALID.zip
[원천]EMOIMG_불안_VALID.zip
[원천]EMOIMG_상처_VALID.zip
[원천]EMOIMG_슬픔_VALID.zip
[원천]EMOIMG_중립_VALID.zip


In [27]:
# Validation 원천 ZIP 7개 압축 해제
for zip_path in sorted(valid_source_zips):

    print("압축 해제 중:", zip_path.name)

    with zipfile.ZipFile(zip_path, "r") as z:

        for member in z.namelist():

            # 깨진 한글 경로와 파일명 복원
            fixed_member = fix_zip_filename(member)

            # 저장할 경로 설정
            target_path = VALID_RAW_DIR / fixed_member

            # 폴더인 경우 생성
            if member.endswith("/"):
                target_path.mkdir(
                    parents=True,
                    exist_ok=True
                )
                continue

            # 파일이 들어갈 상위 폴더 생성
            target_path.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            # ZIP 내부 파일을 실제 파일로 저장
            with z.open(member) as source:
                with open(target_path, "wb") as target:
                    target.write(source.read())

    print("완료:", zip_path.name)

print("Validation 원천 이미지 압축 해제 완료")

압축 해제 중: EMOIMG_기쁨_VALID.zip
완료: EMOIMG_기쁨_VALID.zip
압축 해제 중: [원천]EMOIMG_당황_VALID.zip
완료: [원천]EMOIMG_당황_VALID.zip
압축 해제 중: [원천]EMOIMG_분노_VALID.zip
완료: [원천]EMOIMG_분노_VALID.zip
압축 해제 중: [원천]EMOIMG_불안_VALID.zip
완료: [원천]EMOIMG_불안_VALID.zip
압축 해제 중: [원천]EMOIMG_상처_VALID.zip
완료: [원천]EMOIMG_상처_VALID.zip
압축 해제 중: [원천]EMOIMG_슬픔_VALID.zip
완료: [원천]EMOIMG_슬픔_VALID.zip
압축 해제 중: [원천]EMOIMG_중립_VALID.zip
완료: [원천]EMOIMG_중립_VALID.zip
Validation 원천 이미지 압축 해제 완료


In [28]:
# Validation 원천 이미지 압축 해제 결과 검증

image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}

valid_image_count = sum(
    1
    for path in VALID_RAW_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in image_extensions
)

print("Validation 이미지 수:", valid_image_count)

Validation 이미지 수: 52126


### 2-6. Validation 라벨 압축 해제 및 검증

Validation 데이터의 7개 감정 클래스에 해당하는 라벨 ZIP을 찾아 압축 해제한다.

기쁨 Validation 라벨 ZIP은 다른 클래스와 달리 별도 폴더에 저장되어 있으므로,
해당 경로 예외를 포함하여 7개의 라벨 ZIP을 모두 선택한다.

압축 해제 후 각 JSON 파일의 레코드 수를 확인하고,
전체 Validation 이미지 수인 52,126건과 일치하는지 검증한다.

In [30]:
# Validation 라벨 ZIP 7개 선택
valid_label_zips = [
    path for path in all_zip_files
    if (
        "VALID" in path.name
        and (
            "[라벨]" in path.name
            or "라벨링데이터_231004_add" in str(path.parent)
        )
    )
]

print("Validation 라벨 ZIP 수:", len(valid_label_zips))

for path in sorted(valid_label_zips):
    print(path)

Validation 라벨 ZIP 수: 7
D:\emotion_recognition_project\02_data\downloads\092.한국인 감정인식을 위한 복합 영상 데이터\01.데이터\2.Validation\라벨링데이터_231004_add\EMOIMG_기쁨_VALID.zip
D:\emotion_recognition_project\02_data\downloads\한국인 감정인식을 위한 복합 영상\Validation\[라벨]EMOIMG_당황_VALID.zip
D:\emotion_recognition_project\02_data\downloads\한국인 감정인식을 위한 복합 영상\Validation\[라벨]EMOIMG_분노_VALID.zip
D:\emotion_recognition_project\02_data\downloads\한국인 감정인식을 위한 복합 영상\Validation\[라벨]EMOIMG_불안_VALID.zip
D:\emotion_recognition_project\02_data\downloads\한국인 감정인식을 위한 복합 영상\Validation\[라벨]EMOIMG_상처_VALID.zip
D:\emotion_recognition_project\02_data\downloads\한국인 감정인식을 위한 복합 영상\Validation\[라벨]EMOIMG_슬픔_VALID.zip
D:\emotion_recognition_project\02_data\downloads\한국인 감정인식을 위한 복합 영상\Validation\[라벨]EMOIMG_중립_VALID.zip


#### Validation 라벨 ZIP 압축 해제

확인된 7개의 Validation 라벨 ZIP을 `02_data/labels/valid` 폴더에 압축 해제한다.

각 ZIP 내부에는 감정별 JSON 파일이 바로 들어 있으므로,
압축 해제 후 `labels/valid` 폴더 아래에 7개의 JSON 파일이 저장되도록 한다.

한글 파일명이 깨질 수 있으므로 `fix_zip_filename()` 함수를 적용한다.

In [31]:
# Validation 라벨 ZIP 7개 압축 해제
for zip_path in sorted(valid_label_zips):

    print("압축 해제 중:", zip_path.name)

    with zipfile.ZipFile(zip_path, "r") as z:

        for member in z.namelist():

            # 깨진 한글 파일명 복원
            fixed_member = fix_zip_filename(member)

            # labels/valid 아래에 저장
            target_path = VALID_LABEL_DIR / fixed_member

            # 폴더인 경우 생성
            if member.endswith("/"):
                target_path.mkdir(
                    parents=True,
                    exist_ok=True
                )
                continue

            # 상위 폴더 생성
            target_path.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            # ZIP 내부 파일 저장
            with z.open(member) as source:
                with open(target_path, "wb") as target:
                    target.write(source.read())

    print("완료:", zip_path.name)

print("Validation 라벨 압축 해제 완료")

압축 해제 중: EMOIMG_기쁨_VALID.zip
완료: EMOIMG_기쁨_VALID.zip
압축 해제 중: [라벨]EMOIMG_당황_VALID.zip
완료: [라벨]EMOIMG_당황_VALID.zip
압축 해제 중: [라벨]EMOIMG_분노_VALID.zip
완료: [라벨]EMOIMG_분노_VALID.zip
압축 해제 중: [라벨]EMOIMG_불안_VALID.zip
완료: [라벨]EMOIMG_불안_VALID.zip
압축 해제 중: [라벨]EMOIMG_상처_VALID.zip
완료: [라벨]EMOIMG_상처_VALID.zip
압축 해제 중: [라벨]EMOIMG_슬픔_VALID.zip
완료: [라벨]EMOIMG_슬픔_VALID.zip
압축 해제 중: [라벨]EMOIMG_중립_VALID.zip
완료: [라벨]EMOIMG_중립_VALID.zip
Validation 라벨 압축 해제 완료


#### Validation 라벨 압축 해제 결과 검증

압축 해제된 Validation 라벨 JSON 파일이 7개인지 확인하고,
각 JSON 내부의 레코드 수를 합산하여 전체 Validation 이미지 수 52,126장과 일치하는지 검증한다.

In [32]:
# Validation 라벨 JSON 파일 및 레코드 수 검증

valid_json_files = list(VALID_LABEL_DIR.rglob("*.json"))

print("Validation 라벨 JSON 파일 수:", len(valid_json_files))

valid_total_records = 0

for json_path in sorted(valid_json_files):

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    record_count = len(data)
    valid_total_records += record_count

    print(
        json_path.name,
        ":",
        f"{record_count:,}건"
    )

print(
    "Validation 전체 라벨 레코드:",
    f"{valid_total_records:,}건"
)

Validation 라벨 JSON 파일 수: 7
img_emotion_validation_data(기쁨).json : 7,499건
img_emotion_validation_data(당황).json : 7,454건
img_emotion_validation_data(분노).json : 7,461건
img_emotion_validation_data(불안).json : 7,407건
img_emotion_validation_data(상처).json : 7,423건
img_emotion_validation_data(슬픔).json : 7,479건
img_emotion_validation_data(중립).json : 7,403건
Validation 전체 라벨 레코드: 52,126건


## 3. 실제 사용할 이미지와 라벨 데이터 구성

압축 해제된 원천 이미지와 라벨 데이터를 연결하여
실제 모델 학습에 사용할 데이터 구조를 구성한다.

Training 데이터는 전체 원천데이터 중 `TRAIN_01`과 `TRAIN_02`만 사용하므로,
전체 Training 라벨 417,167건 중 실제 존재하는 이미지 223,578장에 해당하는
라벨 정보만 선택한다.

이 단계에서는 새로운 감정 라벨을 만들지 않고,
AI-Hub에서 제공한 기존 라벨 정보를 그대로 유지한다.

### 3-1. Training 이미지에 해당하는 라벨 레코드 선택

`raw/train`에 실제로 존재하는 이미지 파일명을 기준으로
전체 Training 라벨에서 사용할 레코드만 선택한다.

In [33]:
# 실제 사용할 Training 이미지 파일명 수집

image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}

train_image_files = [
    path
    for path in TRAIN_RAW_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in image_extensions
]

train_filenames = {
    path.name
    for path in train_image_files
}

print("Training 이미지 파일 수:", len(train_image_files))
print("Training 고유 파일명 수:", len(train_filenames))

Training 이미지 파일 수: 223578
Training 고유 파일명 수: 223578


In [34]:
# 전체 Training 라벨에서 실제 사용할 이미지에 해당하는 레코드만 선택

train_label_records = []

for json_path in sorted(TRAIN_LABEL_DIR.rglob("*.json")):

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for record in data:

        if record["filename"] in train_filenames:
            train_label_records.append(record)

print("선택된 Training 라벨 레코드 수:", len(train_label_records))

선택된 Training 라벨 레코드 수: 223578


### 3-2. Validation 이미지와 라벨 연결 확인

`raw/valid`에 존재하는 이미지 파일명과 Validation 라벨의 `filename`을 비교하여
52,126장의 이미지가 모두 라벨과 정상적으로 연결되는지 확인한다.

In [37]:
# Validation 이미지와 라벨 filename 1:1 대응 확인

# 이미지 파일명 수집
valid_filenames = {
    path.name
    for path in VALID_RAW_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in image_extensions
}

# 라벨 filename 수집
valid_label_filenames = set()

for json_path in sorted(VALID_LABEL_DIR.rglob("*.json")):

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    valid_label_filenames.update(
        record["filename"]
        for record in data
    )

# 서로 대응하지 않는 항목 확인
missing_labels = valid_filenames - valid_label_filenames
extra_labels = valid_label_filenames - valid_filenames

print("Validation 이미지 파일명 수:", len(valid_filenames))
print("Validation 라벨 파일명 수:", len(valid_label_filenames))
print("라벨이 없는 이미지 수:", len(missing_labels))
print("이미지가 없는 라벨 수:", len(extra_labels))

Validation 이미지 파일명 수: 52126
Validation 라벨 파일명 수: 52126
라벨이 없는 이미지 수: 0
이미지가 없는 라벨 수: 0


### 3단계 결과

- Training 원천 이미지 223,578장에 해당하는 라벨 레코드 223,578건을 선택하였다.
- Validation 이미지 52,126장과 라벨 52,126건이 파일명 기준으로 모두 1:1 대응함을 확인하였다.
- 따라서 실제 학습과 검증에 사용할 이미지-라벨 구성이 정상적으로 완료되었다.

## 4. 이미지 전처리

모델 입력 크기에 맞추기 위해 Training과 Validation 이미지를 224×224 크기로 변환한다.

원본 데이터는 유지하고, 전처리된 이미지는 `02_data/processed` 폴더에 별도로 저장한다.

In [38]:
# 전처리 이미지 저장 폴더 생성
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_TRAIN_DIR = PROCESSED_DIR / "train"
PROCESSED_VALID_DIR = PROCESSED_DIR / "valid"

PROCESSED_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_VALID_DIR.mkdir(parents=True, exist_ok=True)

In [40]:
import numpy as np
import cv2

# 이미지 크기를 224x224로 변환하여 저장하는 함수
def resize_and_save_images(source_dir, target_dir):

    image_paths = [
        path
        for path in source_dir.rglob("*")
        if path.is_file()
        and path.suffix.lower() in image_extensions
    ]

    for path in image_paths:

        # 이미지 불러오기
        image_bytes = np.fromfile(str(path), dtype=np.uint8)
        image = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)

        if image is None:
            continue

        # 224x224로 크기 변환
        resized = cv2.resize(image, (224, 224))

        # 원래 파일명 그대로 저장
        save_path = target_dir / path.name

        success, encoded = cv2.imencode(path.suffix, resized)

        if success:
            encoded.tofile(str(save_path))

### 4-2. Training 이미지 224×224 변환

Training 원본 이미지를 224×224 크기로 변환하여
`processed/train` 폴더에 저장한다.

In [ ]:
# Training 이미지 224x224 변환
resize_and_save_images(
    TRAIN_RAW_DIR,
    PROCESSED_TRAIN_DIR
)

print("Training 이미지 전처리 완료")

In [2]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\emotion_recognition_project")
DATA_DIR = PROJECT_ROOT / "02_data"

PROCESSED_TRAIN_DIR = DATA_DIR / "processed" / "train"

image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}

In [3]:
# 어제까지 변환되어 저장된 Training 이미지 수 확인
processed_train_count = sum(
    1
    for path in PROCESSED_TRAIN_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in image_extensions
)

print("현재 전처리된 Training 이미지 수:", processed_train_count)

현재 전처리된 Training 이미지 수: 216807


In [4]:
import numpy as np
import cv2

# 원본 Training 경로 다시 설정
TRAIN_RAW_DIR = DATA_DIR / "raw" / "train"

def resize_remaining_images(source_dir, target_dir):

    for path in source_dir.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in image_extensions:
            continue

        save_path = target_dir / path.name

        # 이미 전처리된 파일은 건너뜀
        if save_path.exists():
            continue

        # 원본 이미지 불러오기
        image_bytes = np.fromfile(str(path), dtype=np.uint8)
        image = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)

        if image is None:
            continue

        # 224x224로 변환
        resized = cv2.resize(image, (224, 224))

        # 같은 파일명으로 저장
        success, encoded = cv2.imencode(path.suffix, resized)

        if success:
            encoded.tofile(str(save_path))

print("남은 Training 이미지 변환 준비 완료")

남은 Training 이미지 변환 준비 완료


In [5]:
# 남은 Training 이미지 224x224 변환
resize_remaining_images(
    TRAIN_RAW_DIR,
    PROCESSED_TRAIN_DIR
)

print("Training 이미지 전처리 완료")

Training 이미지 전처리 완료


### 4-3. Training 전처리 결과 검증

변환된 Training 이미지 수가 원본과 일치하는지 확인하고,
일부 이미지를 직접 불러와 손상 여부와 224×224 크기 변환 결과를 검증한다.

In [6]:
# 전처리된 Training 이미지 수 확인
processed_train_files = [
    path
    for path in PROCESSED_TRAIN_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in image_extensions
]

print("원본 Training 이미지 수:", 223578)
print("전처리 Training 이미지 수:", len(processed_train_files))

원본 Training 이미지 수: 223578
전처리 Training 이미지 수: 223578


In [7]:
import random

# 전처리 이미지 중 100장 무작위 선택
sample_files = random.sample(processed_train_files, 100)

damaged_count = 0
wrong_size_count = 0

for path in sample_files:

    # 이미지 불러오기
    image_bytes = np.fromfile(str(path), dtype=np.uint8)
    image = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)

    # 파일을 읽지 못하면 손상된 것으로 확인
    if image is None:
        damaged_count += 1
        continue

    # 이미지 크기가 224x224인지 확인
    if image.shape[:2] != (224, 224):
        wrong_size_count += 1

print("검사한 이미지 수:", len(sample_files))
print("읽기 실패 이미지 수:", damaged_count)
print("224x224가 아닌 이미지 수:", wrong_size_count)

검사한 이미지 수: 100
읽기 실패 이미지 수: 0
224x224가 아닌 이미지 수: 0


### 4-4. Validation 이미지 224×224 변환

`02_data/raw/valid`의 Validation 원본 이미지를 224×224로 변환하여
`02_data/processed/valid`에 저장한다.

In [10]:
import numpy as np
import cv2

# 프로젝트 기준 경로
VALID_RAW_DIR = DATA_DIR / "raw" / "valid"
PROCESSED_VALID_DIR = DATA_DIR / "processed" / "valid"

# 이미지 크기를 224x224로 변환하여 저장하는 함수
def resize_and_save_images(source_dir, target_dir):

    image_paths = [
        path
        for path in source_dir.rglob("*")
        if path.is_file()
        and path.suffix.lower() in image_extensions
    ]

    for path in image_paths:

        # 원본 이미지 불러오기
        image_bytes = np.fromfile(str(path), dtype=np.uint8)
        image = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)

        if image is None:
            continue

        # 224x224로 크기 변환
        resized = cv2.resize(image, (224, 224))

        # processed/valid에 같은 파일명으로 저장
        save_path = target_dir / path.name

        success, encoded = cv2.imencode(path.suffix, resized)

        if success:
            encoded.tofile(str(save_path))

# Validation 이미지 변환 실행
resize_and_save_images(
    VALID_RAW_DIR,
    PROCESSED_VALID_DIR
)

print("Validation 이미지 전처리 완료")

Validation 이미지 전처리 완료


## 5. 전처리 결과 검증 및 저장

Training과 Validation의 전처리 이미지 수와 이미지 크기를 확인하여
224×224 변환이 정상적으로 완료되었는지 최종 검증한다.

In [11]:
# 전처리된 Validation 이미지 수 확인
processed_valid_files = [
    path
    for path in PROCESSED_VALID_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in image_extensions
]

print("원본 Validation 이미지 수:", 52126)
print("전처리 Validation 이미지 수:", len(processed_valid_files))

원본 Validation 이미지 수: 52126
전처리 Validation 이미지 수: 52126


In [12]:
# Training / Validation 전처리 이미지 크기 확인
train_sample = cv2.imdecode(
    np.fromfile(str(processed_train_files[0]), dtype=np.uint8),
    cv2.IMREAD_COLOR
)

valid_sample = cv2.imdecode(
    np.fromfile(str(processed_valid_files[0]), dtype=np.uint8),
    cv2.IMREAD_COLOR
)

print("Training 이미지 shape:", train_sample.shape)
print("Validation 이미지 shape:", valid_sample.shape)

Training 이미지 shape: (224, 224, 3)
Validation 이미지 shape: (224, 224, 3)


### 5단계 결과

- Training 전처리 이미지 223,578장이 정상 저장되었음을 확인하였다.
- Validation 전처리 이미지 52,126장이 정상 저장되었음을 확인하였다.
- Training과 Validation 샘플 이미지의 크기는 모두 `(224, 224, 3)`으로 확인되었다.
- 따라서 224×224 이미지 전처리를 완료하였다.

## 6. 흑백 변환 및 PNG 저장

팀 회의 결과에 따라 기존 224×224 전처리 이미지를 흑백으로 변환하고,
파일 형식을 PNG로 통일하여 별도 폴더에 저장한다.

기존 전처리 결과는 유지하여 이전 작업 내용을 보존한다.

### 6-1. 흑백 PNG 저장 경로 설정

In [14]:
from pathlib import Path

# 프로젝트 기준 경로
PROJECT_ROOT = Path(r"D:\emotion_recognition_project")
DATA_DIR = PROJECT_ROOT / "02_data"

# 기존 전처리 결과 경로
PROCESSED_DIR = DATA_DIR / "processed"

# 흑백 PNG 저장 경로
GRAYSCALE_DIR = PROCESSED_DIR / "grayscale_png"

GRAY_TRAIN_DIR = GRAYSCALE_DIR / "train"
GRAY_VALID_DIR = GRAYSCALE_DIR / "valid"

GRAY_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
GRAY_VALID_DIR.mkdir(parents=True, exist_ok=True)

print("Training 저장 경로:", GRAY_TRAIN_DIR)
print("Validation 저장 경로:", GRAY_VALID_DIR)

Training 저장 경로: D:\emotion_recognition_project\02_data\processed\grayscale_png\train
Validation 저장 경로: D:\emotion_recognition_project\02_data\processed\grayscale_png\valid


### 6-2. 흑백 PNG 변환 함수 정의

기존 224×224 컬러 이미지를 흑백으로 변환하고,
파일명은 유지한 채 PNG 형식으로 저장하는 함수를 정의한다.

In [15]:
import numpy as np
import cv2

# 기존 224x224 컬러 이미지를 흑백 PNG로 변환하여 저장하는 함수
def convert_to_grayscale_png(source_dir, target_dir):

    image_files = [
        path
        for path in source_dir.rglob("*")
        if path.is_file()
        and path.suffix.lower() in image_extensions
    ]

    for path in image_files:

        # 기존 전처리 이미지 불러오기
        image_bytes = np.fromfile(str(path), dtype=np.uint8)
        image = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)

        if image is None:
            continue

        # 컬러 이미지를 흑백으로 변환
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # 파일명은 유지하고 확장자만 .png로 변경
        save_path = target_dir / f"{path.stem}.png"

        success, encoded = cv2.imencode(".png", gray)

        if success:
            encoded.tofile(str(save_path))

### 6-3. Training / Validation 이미지 흑백 PNG 변환

기존 224×224 Training과 Validation 이미지를 흑백으로 변환하고,
PNG 형식으로 저장한다.

In [16]:
# 기존 224x224 이미지 경로
PROCESSED_TRAIN_DIR = PROCESSED_DIR / "train"
PROCESSED_VALID_DIR = PROCESSED_DIR / "valid"

# Training 변환
convert_to_grayscale_png(
    PROCESSED_TRAIN_DIR,
    GRAY_TRAIN_DIR
)
print("Training 흑백 PNG 변환 완료")

# Validation 변환
convert_to_grayscale_png(
    PROCESSED_VALID_DIR,
    GRAY_VALID_DIR
)
print("Validation 흑백 PNG 변환 완료")

Training 흑백 PNG 변환 완료
Validation 흑백 PNG 변환 완료


### 6-4. 흑백 PNG 전처리 결과 검증

Training과 Validation 이미지가 모두 PNG 형식으로 저장되었는지 확인하고,
샘플 이미지의 크기가 224×224 흑백 이미지인지 검증한다.

In [1]:
from pathlib import Path
import numpy as np
import cv2

# --------------------------------------------------
# 1. 프로젝트 및 흑백 PNG 저장 경로 설정
# --------------------------------------------------

# 프로젝트 최상위 경로
PROJECT_ROOT = Path(r"D:\emotion_recognition_project")

# 흑백 PNG로 변환된 Training 이미지 폴더
GRAY_TRAIN_DIR = (
    PROJECT_ROOT
    / "02_data"
    / "processed"
    / "grayscale_png"
    / "train"
)

# 흑백 PNG로 변환된 Validation 이미지 폴더
GRAY_VALID_DIR = (
    PROJECT_ROOT
    / "02_data"
    / "processed"
    / "grayscale_png"
    / "valid"
)


# --------------------------------------------------
# 2. PNG 파일 목록 불러오기
# --------------------------------------------------

# Training 폴더 안의 모든 PNG 파일 경로를 리스트로 저장
gray_train_files = list(GRAY_TRAIN_DIR.glob("*.png"))

# Validation 폴더 안의 모든 PNG 파일 경로를 리스트로 저장
gray_valid_files = list(GRAY_VALID_DIR.glob("*.png"))


# --------------------------------------------------
# 3. 전처리된 이미지 개수 확인
# --------------------------------------------------

# Training은 실제 사용하는 원천 이미지 223,578장과 같아야 함
print("Training 흑백 PNG 이미지 수:", len(gray_train_files))

# Validation은 전체 52,126장과 같아야 함
print("Validation 흑백 PNG 이미지 수:", len(gray_valid_files))


# --------------------------------------------------
# 4. Training 샘플 이미지 1장 읽기
# --------------------------------------------------

# Windows 한글 경로 문제를 피하기 위해
# np.fromfile()로 이미지 파일을 먼저 바이트 형태로 읽음
train_image_bytes = np.fromfile(
    str(gray_train_files[0]),
    dtype=np.uint8
)

# 바이트 데이터를 실제 이미지 배열로 변환
# IMREAD_GRAYSCALE을 사용하여 흑백 이미지로 읽음
train_sample = cv2.imdecode(
    train_image_bytes,
    cv2.IMREAD_GRAYSCALE
)


# --------------------------------------------------
# 5. Validation 샘플 이미지 1장 읽기
# --------------------------------------------------

valid_image_bytes = np.fromfile(
    str(gray_valid_files[0]),
    dtype=np.uint8
)

valid_sample = cv2.imdecode(
    valid_image_bytes,
    cv2.IMREAD_GRAYSCALE
)


# --------------------------------------------------
# 6. 이미지 크기 및 흑백 여부 확인
# --------------------------------------------------

# 정상적인 흑백 224x224 이미지라면
# shape이 (224, 224) 형태로 출력되어야 함
print("Training 샘플 shape:", train_sample.shape)
print("Validation 샘플 shape:", valid_sample.shape)

# 흑백 이미지는 2차원 배열이므로 ndim 값이 2가 되어야 함
print("Training 샘플 차원 수:", train_sample.ndim)
print("Validation 샘플 차원 수:", valid_sample.ndim)

Training 흑백 PNG 이미지 수: 223578
Validation 흑백 PNG 이미지 수: 52126
Training 샘플 shape: (224, 224)
Validation 샘플 shape: (224, 224)
Training 샘플 차원 수: 2
Validation 샘플 차원 수: 2


### 6단계 결과

- Training 이미지 223,578장을 흑백 PNG 형식으로 변환하였다.
- Validation 이미지 52,126장을 흑백 PNG 형식으로 변환하였다.
- Training과 Validation 샘플 이미지의 크기는 모두 `(224, 224)`로 확인되었다.
- 샘플 이미지의 차원 수가 2로 확인되어 흑백 변환이 정상적으로 적용되었음을 검증하였다.
- 최종 전처리 데이터는 224×224 크기의 흑백 PNG 이미지로 구성하였다.